# Run the RhyME Bioconductor tests yourself

Run this notebook from top to bottom with the **R 4.6.1 (Bioconductor 3.23)** kernel. It performs the real package workflow rather than reading previously saved logs:

1. `R CMD build` — builds the source tarball and **knits the vignette to HTML**;
2. extracts the knitted HTML vignette into the run directory;
3. `R CMD check` — installs the package, runs examples and unit tests, and rebuilds the vignette;
4. `BiocCheck` — runs the additional Bioconductor submission checks;
5. summarizes the new logs.

Each run uses a new timestamped directory under `outputs/`, so existing results are not overwritten. The current source contains 10 test functions covering fixed and mixed fits, robust automatic-prior estimation, covariate adjustment, returned summaries, plotting defaults/marginal curves, and input validation.

In [ ]:
version_or_missing <- function(package) {
    if (requireNamespace(package, quietly = TRUE)) {
        as.character(utils::packageVersion(package))
    } else {
        "not installed"
    }
}

environment <- data.frame(
    component = c("R", "Bioconductor", "TMB", "BiocCheck", "BiocStyle"),
    version = c(
        paste(R.version$major, R.version$minor, sep = "."),
        as.character(BiocManager::version()),
        version_or_missing("TMB"),
        version_or_missing("BiocCheck"),
        version_or_missing("BiocStyle")
    )
)
environment

R_BIN <- normalizePath(file.path(R.home("bin"), "R"))
R_PREFIX_BIN <- normalizePath(
    file.path(R.home(), "..", "..", "bin"), mustWork = FALSE
)

required <- c("TMB", "BiocCheck", "BiocStyle", "rmarkdown", "knitr")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
    stop("Missing required packages: ", paste(missing, collapse = ", "))
}

## Expose Pandoc to the R CMD build/check subprocesses.
pandoc_candidates <- c(
    unname(Sys.which("pandoc")),
    file.path(path.expand("~"), ".local", "bin", "pandoc"),
    file.path(path.expand("~"), "miniconda3", "bin", "pandoc")
)
pandoc_candidates <- unique(pandoc_candidates[nzchar(pandoc_candidates)])
pandoc_candidates <- pandoc_candidates[
    file.exists(pandoc_candidates) & file.access(pandoc_candidates, 1L) == 0L
]
if (!length(pandoc_candidates)) {
    stop("Pandoc is unavailable; install it before building the vignette.",
         call. = FALSE)
}
PANDOC_BIN <- normalizePath(pandoc_candidates[[1]])
Sys.setenv(
    PATH = paste(
        R_PREFIX_BIN, dirname(R_BIN), dirname(PANDOC_BIN),
        Sys.getenv("PATH"),
        sep = .Platform$path.sep
    ),
    RSTUDIO_PANDOC = dirname(PANDOC_BIN)
)
if (!rmarkdown::pandoc_available()) {
    stop("rmarkdown still cannot detect Pandoc at ", PANDOC_BIN,
         call. = FALSE)
}
cat("Pandoc:", PANDOC_BIN,
    "(version", as.character(rmarkdown::pandoc_version()), ")\n")

cxx17 <- trimws(system2(R_BIN, c("CMD", "config", "CXX17"), stdout = TRUE))
compiler <- strsplit(cxx17[[1]], "[[:space:]]+")[[1]][1]
if (!nzchar(Sys.which(compiler))) {
    stop("C++17 compiler is unavailable: ", compiler, call. = FALSE)
}
cat("Compiler:", normalizePath(Sys.which(compiler)), "\n")

## 1. Configure a fresh run

`SKIP_DEPRECATED_CHECK` is currently `TRUE` because BiocCheck 1.48.1 fails during its own remote deprecated-package cache lookup on this machine. Set it to `FALSE` to retry the complete lookup if that upstream problem has been resolved. All other BiocCheck rules still run.

In [ ]:
source_candidates <- unique(normalizePath(
    c(getwd(), file.path(getwd(), "RhyME"), dirname(getwd())),
    mustWork = FALSE
))
is_package <- file.exists(file.path(source_candidates, "DESCRIPTION"))
if (!any(is_package)) {
    stop("Run this notebook from the RhyME repository or its notebooks directory.")
}
PACKAGE_DIR <- source_candidates[which(is_package)[1L]]
PROJECT_DIR <- dirname(PACKAGE_DIR)
RUN_ID <- format(Sys.time(), "%Y%m%d_%H%M%S")
RUN_DIR <- file.path(PROJECT_DIR, "outputs", paste0("RhyME_Bioc_test_", RUN_ID))
CHECK_ARGS <- c("--no-manual", "--timings")
SKIP_DEPRECATED_CHECK <- TRUE

UNIT_TEST_FILE <- file.path(PACKAGE_DIR, "inst", "unitTests", "test_RhyME.R")
test_source <- readLines(UNIT_TEST_FILE, warn = FALSE)
test_functions <- grep(
    "^test_RhyME_[[:alnum:]_]+[[:space:]]*<-[[:space:]]*function",
    test_source, value = TRUE
)
required_tests <- c(
    "test_RhyME_covariate_adjustment",
    "test_RhyME_plot_uses_fitted_design",
    "test_RhyME_input_validation"
)
stopifnot(
    dir.exists(PACKAGE_DIR),
    file.exists(file.path(PACKAGE_DIR, "DESCRIPTION")),
    file.exists(UNIT_TEST_FILE),
    length(test_functions) == 10L,
    all(vapply(required_tests, function(x) any(grepl(x, test_source)),
               logical(1)))
)
dir.create(RUN_DIR, recursive = TRUE, showWarnings = FALSE)

data.frame(
    item = c("Package source", "Fresh output directory", "R executable"),
    value = c(PACKAGE_DIR, RUN_DIR, R_BIN)
)

In [ ]:
run_logged <- function(args, log_file, wd = RUN_DIR, env = character(),
                       stop_on_error = TRUE) {
    previous <- setwd(wd)
    on.exit(setwd(previous), add = TRUE)

    message("Running: R ", paste(args, collapse = " "))
    status <- system2(
        R_BIN, args, stdout = log_file, stderr = log_file, env = env
    )
    lines <- readLines(log_file, warn = FALSE)
    cat(paste(utils::tail(lines, 35), collapse = "\n"), "\n")

    if (stop_on_error && status != 0L) {
        stop("Command failed; inspect ", log_file, call. = FALSE)
    }
    invisible(status)
}

## 2. Build the source package

This is the first long-running cell. `R CMD build` installs RhyME temporarily and executes the vignette, so a successful build confirms that the vignette can knit.

In [ ]:
BUILD_LOG <- file.path(RUN_DIR, "build.log")
build_status <- run_logged(
    c("CMD", "build", PACKAGE_DIR),
    BUILD_LOG
)

TARBALL <- list.files(
    RUN_DIR, pattern = "^RhyME_.*[.]tar[.]gz$", full.names = TRUE
)
stopifnot(length(TARBALL) == 1L, file.exists(TARBALL))
TARBALL

## 3. Extract the knitted HTML vignette

The HTML is already knitted inside the source tarball by the preceding build. This cell extracts that exact build artifact rather than fitting the vignette models a second time.

In [ ]:
VIGNETTE_DIR <- file.path(RUN_DIR, "vignette")
VIGNETTE_EXTRACT_DIR <- file.path(RUN_DIR, "vignette_extract")
dir.create(VIGNETTE_DIR)
dir.create(VIGNETTE_EXTRACT_DIR)

utils::untar(
    TARBALL,
    files = "RhyME/inst/doc/RhyME_vignette.html",
    exdir = VIGNETTE_EXTRACT_DIR
)
VIGNETTE_HTML <- file.path(VIGNETTE_DIR, "RHyME_vignette.html")
copied <- file.copy(
    file.path(VIGNETTE_EXTRACT_DIR, "RhyME", "inst", "doc",
              "RhyME_vignette.html"),
    VIGNETTE_HTML, overwrite = TRUE
)
stopifnot(copied, file.exists(VIGNETTE_HTML))

data.frame(
    html = normalizePath(VIGNETTE_HTML),
    size_mb = round(file.info(VIGNETTE_HTML)$size / 1024^2, 2)
)

## 4. Run `R CMD check`

This runs installation, namespace and documentation checks, examples, the RUnit test suite, vignette checks, and a fresh vignette rebuild. `--no-manual` skips only the PDF reference manual because a LaTeX toolchain is not part of this R environment. Remove it from `CHECK_ARGS` if you also want to test manual generation.

In [ ]:
CHECK_COMMAND_LOG <- file.path(RUN_DIR, "check-command.log")
check_status <- run_logged(
    c("CMD", "check", CHECK_ARGS, basename(TARBALL)),
    CHECK_COMMAND_LOG,
    env = "_R_CHECK_FORCE_SUGGESTS_=true"
)

CHECK_DIR <- file.path(RUN_DIR, "RhyME.Rcheck")
CHECK_LOG <- file.path(CHECK_DIR, "00check.log")
RUNIT_LOG <- file.path(CHECK_DIR, "tests", "runTest.Rout")
stopifnot(file.exists(CHECK_LOG), file.exists(RUNIT_LOG))
runit_summary <- grep(
    "RhyME RUnit Tests -", readLines(RUNIT_LOG, warn = FALSE),
    value = TRUE
)
stopifnot(
    length(runit_summary) == 1L,
    grepl("10 test functions, 0 errors, 0 failures", runit_summary)
)
list(
    R_CMD_check = grep("^Status:", readLines(CHECK_LOG, warn = FALSE),
                       value = TRUE),
    unit_tests = runit_summary
)

## 5. Run `BiocCheck`

BiocCheck includes submission-specific metadata, style, account, and repository checks. A Support Site registration error is reported separately from package compilation/tests. This cell does not stop the notebook on a nonzero BiocCheck exit status, so the summary cell can still display the complete findings.

In [ ]:
BIOC_DIR <- file.path(RUN_DIR, "BiocCheck")
dir.create(BIOC_DIR)
BIOC_COMMAND_LOG <- file.path(RUN_DIR, "bioccheck-command.log")

bioc_expression <- sprintf(
    paste0(
        "BiocCheck::BiocCheck(package=%s, checkDir=%s, callr=FALSE, ",
        "`no-check-deprecated`=%s)"
    ),
    deparse(TARBALL),
    deparse(BIOC_DIR),
    if (SKIP_DEPRECATED_CHECK) "TRUE" else "FALSE"
)

bioc_status <- run_logged(
    c("--vanilla", "-e", shQuote(bioc_expression)),
    BIOC_COMMAND_LOG,
    stop_on_error = FALSE
)
bioc_status

## 6. Summarize this run

In [ ]:
check_lines <- readLines(CHECK_LOG, warn = FALSE)
runit_lines <- readLines(RUNIT_LOG, warn = FALSE)
bioc_lines <- readLines(BIOC_COMMAND_LOG, warn = FALSE)
r_check_result <- grep("^Status:", check_lines, value = TRUE)
runit_result <- grep("RhyME RUnit Tests -", runit_lines, value = TRUE)
bioc_result <- grep("ERRORS.*WARNINGS.*NOTES", bioc_lines, value = TRUE)

summary <- data.frame(
    stage = c("R CMD build", "HTML vignette", "RUnit tests",
              "R CMD check", "BiocCheck"),
    result = c(
        if (build_status == 0L) "PASS" else paste("exit", build_status),
        if (file.exists(VIGNETTE_HTML)) "CREATED" else "MISSING",
        if (length(runit_result)) runit_result[[1]] else "MISSING",
        if (length(r_check_result)) r_check_result[[1]] else paste("exit", check_status),
        if (length(bioc_result)) sub("^[^0-9]*", "", bioc_result[[1]]) else paste("exit", bioc_status)
    ),
    output = c(BUILD_LOG, VIGNETTE_HTML, RUNIT_LOG, CHECK_LOG,
               BIOC_COMMAND_LOG)
)
summary

In [ ]:
cat(
    "Fresh run directory:\n", normalizePath(RUN_DIR),
    "\n\nKnitted HTML vignette:\n", normalizePath(VIGNETTE_HTML),
    "\n\nR CMD check log:\n", normalizePath(CHECK_LOG),
    "\n\nBiocCheck output:\n", normalizePath(BIOC_DIR),
    "\n", sep = ""
)